### we'll implement the Streaming side of the architecture:

```text
JSON Events
     │
     │ Auto Loader
     ↓
Bronze Delta
```
Instead of repeatedly scanning the entire directory, Auto Loader tracks newly arriving files and process using [.format("cloudFiles")]

In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Source location

source_path = (
    "/Volumes/retail_lakehouse/raw/retail_files/events/"
)

# 2. Schema location

schema_path = (
    "/Volumes/retail_lakehouse/raw/retail_files/"
    "schema/events/"
)

# 3. Checkpoint location

checkpoint_path = (
    "/Volumes/retail_lakehouse/raw/retail_files/"
    "checkpoints/events/"
)

# 4. Read JSON using Auto Loader

events_df = (
    spark.readStream
         .format("cloudFiles")  # this tells databricks to use Auto Loader to detect and ingest files arriving in this location.
         .option("cloudFiles.format", "json")
         .option("cloudFiles.schemaLocation", schema_path) # Auto Loader stores schema information here.
         .load(source_path)
)

In [0]:
# 5. Add ingestion timestamp

events_df = events_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

In [0]:
# 6. Write to Bronze Delta

streaming_query = (
    events_df.writeStream
             .format("delta")
             .option(
                 "checkpointLocation",
                 checkpoint_path
             )                        # The checkpoint stores streaming progress information
             .outputMode("append")
             .trigger(availableNow=True)
             .toTable(
                 "retail_lakehouse.bronze.order_events"
             )
)

# Wait for the streaming query to finish processing all available files
streaming_query.awaitTermination()

**Verify Bronze Streaming Table**

In [0]:
%sql
SELECT COUNT(*) AS total_events
FROM retail_lakehouse.bronze.order_events;

total_events
1500


In [0]:
%sql
SELECT *
FROM retail_lakehouse.bronze.order_events
LIMIT 10;

customer_id,event_id,event_time,event_type,order_id,product_id,quantity,unit_price,_rescued_data,ingestion_time
1024,E00501,2026-08-04,ORDER_CREATED,20501,P027,5,33.26,null,2026-08-29T12:01:05.311Z
1027,E00502,2026-08-27,ORDER_UPDATED,20502,P021,2,158.75,null,2026-08-29T12:01:05.311Z
1054,E00503,2026-08-02,ORDER_CREATED,20503,P022,5,210.73,null,2026-08-29T12:01:05.311Z
1048,E00504,2026-08-16,ORDER_CREATED,20504,P001,1,302.22,null,2026-08-29T12:01:05.311Z
1004,E00505,2026-08-11,ORDER_CREATED,20505,P042,1,341.09,null,2026-08-29T12:01:05.311Z
1015,E00506,2026-08-17,ORDER_UPDATED,20506,P042,3,419.73,null,2026-08-29T12:01:05.311Z
1058,E00507,2026-08-20,ORDER_CREATED,20507,P009,3,165.58,null,2026-08-29T12:01:05.311Z
1051,E00508,2026-08-26,ORDER_CREATED,20508,P017,5,89.44,null,2026-08-29T12:01:05.311Z
1022,E00509,2026-08-06,ORDER_CREATED,20509,P012,3,101.61,null,2026-08-29T12:01:05.311Z
1169,E00510,2026-08-05,ORDER_CREATED,20510,P041,2,122.08,null,2026-08-29T12:01:05.311Z
